# Lab | Transformers
## Route every message to the right team

Yesterday you built a semantic search for a digital bank's support messages. The bank now wants to go one step further: **route** each new message automatically to the team that handles its intent, before any person reads it.

The support lead has picked the ten intents that agents mix up most often: card delivery, lost and compromised cards, PINs, failed top-ups and exchange rates. If a model can tell these apart, the rest will be easy. Your job is to compare three ways of doing it, and to recommend one:

+ a **zero-shot** model, which needs no labelled examples at all;
+ a **fine-tuned DistilBERT**, trained on the bank's labelled messages;
+ yesterday's **semantic search**, used as a classifier: the intent of the most similar past message.

**The data** is the Banking77 dataset from yesterday's lab, in `data/banking_train.csv` and `data/banking_test.csv`, filtered to the ten intents.

**This lab practises** using and fine-tuning pretrained transformers. You will use, from today's **Transformers and Hugging Face** lesson: `AutoTokenizer`, the zero-shot `pipeline`, `Dataset.map`, `AutoModelForSequenceClassification`, `TrainingArguments` and `Trainer`, and the comparison with the methods of Days 1 and 2.

**By the end, you can:**

1. Apply a zero-shot pipeline and fine-tune DistilBERT to classify customer messages by intent.
2. Evaluate both against a semantic-search baseline and justify which one to use.

Each challenge ends with a **checkpoint** that tells you what you should see. If your numbers are very different, fix that before moving on. Fine-tuning takes one to four minutes on a laptop CPU; on a slow computer, open the notebook in [Google Colab](https://colab.research.google.com) with a GPU runtime.

## Setup

You need `transformers`, `torch`, `datasets`, `accelerate` and `sentence-transformers`. If the import fails, install them from a terminal with `pip install transformers torch datasets accelerate sentence-transformers`. The first run downloads three models (about 500 MB in total) unless you already have them from the lesson.

In [6]:
%pip install --upgrade --force-reinstall pyarrow datasets

  Using cached pyarrow-26.0.0-cp314-cp314-macosx_12_0_arm64.whl.metadata (3.0 kB)
  Using cached datasets-5.1.0-py3-none-any.whl.metadata (24 kB)
  Using cached xxhash-4.0.2-cp314-cp314-macosx_11_0_arm64.whl.metadata (17 kB)
  Using cached multiprocess-0.70.19-py314-none-any.whl.metadata (7.2 kB)
  Using cached huggingface_hub-2.3.0-py3-none-any.whl.metadata (16 kB)
  Using cached packaging-26.3-py3-none-any.whl.metadata (3.5 kB)
  Using cached click-8.5.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached hf_xet-1.7.0-cp38-abi3-macosx_11_0_arm64.whl.metadata (4.9 kB)
  Using cached httpx2-2.16.0-py3-none-any.whl.metadata (10 kB)
  Using cached typing_extensions-4.16.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached httpcore2-2.16.0-py3-none-any.whl.metadata (27 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached six-1.17.0-py2.py3-none-any.whl.metadata (1.7 kB)
Using cached pyarrow-26.0.0-cp314

In [1]:
import os
import logging
import warnings
os.environ["HF_HUB_VERBOSITY"] = "error"      # hide Hugging Face download notices
os.environ["TOKENIZERS_PARALLELISM"] = "false"
logging.getLogger("torch.distributed.elastic.multiprocessing.redirects").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message=".*torch.jit.script.*")

import time
import numpy as np
import pandas as pd
import torch

from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from transformers.utils import logging as transformers_logging
transformers_logging.set_verbosity_error()
transformers_logging.disable_progress_bar()
import datasets
datasets.disable_progress_bars()
from sentence_transformers import SentenceTransformer

from sklearn.metrics import accuracy_score, classification_report

pd.set_option("display.max_colwidth", 90)
_ = torch.manual_seed(0)

In [2]:
intents = ["card_arrival", "card_delivery_estimate", "lost_or_stolen_card", "compromised_card", "card_not_working",
           "pin_blocked", "change_pin", "top_up_failed", "top_up_reverted", "exchange_rate"]

train = pd.read_csv("../data/banking_train.csv")
test = pd.read_csv("../data/banking_test.csv")
train = train[train["category"].isin(intents)].reset_index(drop=True)
test = test[test["category"].isin(intents)].reset_index(drop=True)
print(train.shape, test.shape)
train.sample(5, random_state=0)

(1185, 2) (400, 2)


,text,category
833,What do I do if I detect fraudulent use on my car?,compromised_card
567,I lost my card. Can you help me?,lost_or_stolen_card
950,Can my PIN changed remotely without visiting a bank?,change_pin
922,I need to make my card PIN a different number,change_pin
362,how long does it take to get my card i am in a rush,card_delivery_estimate


**Checkpoint:** `(1185, 2) (400, 2)`: 1,185 past messages to learn from, and 400 new messages (40 per intent) to test on.

## Challenge 1: look at the messages as tokens

Before choosing a model, check what the model will actually read.

1. Load the tokenizer of `distilbert/distilbert-base-uncased` and print the tokens of three messages: `"My card hasn't arrived yet"`, `"Someone used my card in Bulgaria!"` and `"Why did my top-up get reverted?"`.
2. Count the tokens of every training message (`len(tokenizer(text)["input_ids"])`) and print the `describe()` of the counts.

In [3]:
checkpoint = "distilbert/distilbert-base-uncased"

# 1. Tokenizer and the tokens of three messages
three_messages = [
    "My card hasn't arrived yet", 
    "Someone used my card in Bulgaria!", 
    "Why did my top-up get reverted?"
    ]
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
for text in three_messages:
    print(f'{text: <35} {tokenizer.tokenize(text)}')

# 2. Number of tokens per training message
token_counts = train['text'].apply(lambda t: len(tokenizer(t)['input_ids']))
token_counts.describe()

My card hasn't arrived yet          ['my', 'card', 'hasn', "'", 't', 'arrived', 'yet']
Someone used my card in Bulgaria!   ['someone', 'used', 'my', 'card', 'in', 'bulgaria', '!']
Why did my top-up get reverted?     ['why', 'did', 'my', 'top', '-', 'up', 'get', 'reverted', '?']


count    1185.000000
mean       15.315612
std         7.985310
min         5.000000
25%        11.000000
50%        13.000000
75%        17.000000
max        66.000000
Name: text, dtype: float64

**Checkpoint:** `hasn't` becomes `hasn`, `'` and `t`, and `top-up` becomes `top`, `-` and `up`. The messages are short: a median of 13 tokens and a mean of about 15, and the longest has 66, including `[CLS]` and `[SEP]`.

In [4]:
(token_counts > 64).sum()

np.int64(1)

**Question:** you will fine-tune with `max_length=64`. How many training messages does that cut, and why is a short `max_length` worth having?

With max_length=64, only the messages longer than 64 tokens are cut (as seen above only 1 here) and they lose just their last one or two tokens, so almost no information is lost. A short limit makes training and prediction much faster and lighter, because attention compares every token with every other one. It also caps the cost of any unusually long message that comes in later.

## Challenge 2: zero-shot routing

The zero-shot model needs no training, only a description of each intent in plain English. The quality of those descriptions matters: the model compares each message with each description.

1. Complete the dictionary `descriptions` with a short description for the last three intents. Seven are given.
2. Create the zero-shot pipeline with the model `MoritzLaurer/deberta-v3-xsmall-zeroshot-v1.1-all-33`.
3. Run it on all the test messages, with `candidate_labels=list(descriptions.values())` and `batch_size=16`, and time it.
4. Turn the top label of each result back into an intent and compute the accuracy.

In [5]:
descriptions = {
    "card_arrival": "my new card has not arrived",
    "card_delivery_estimate": "how long card delivery takes",
    "lost_or_stolen_card": "my card was lost or stolen",
    "compromised_card": "someone else used my card",
    "card_not_working": "my card does not work",
    "pin_blocked": "my PIN is blocked",
    "change_pin": "how to change my PIN",
    'top_up_failed': 'my top-up failed', 
    'top_up_reverted': 'my top-up was reverted', 
    'exchange_rate': 'the exchange rate between currencies'
}

# 2. Zero-shot pipeline
zero_shot = pipeline('zero-shot-classification', model = 'MoritzLaurer/deberta-v3-xsmall-zeroshot-v1.1-all-33')

# 3. Predictions for the test messages, timed
start = time.time()
results = zero_shot(
    test['text'].tolist(), 
    candidate_labels=list(descriptions.values()), 
    batch_size=16
    )
print(f"{len(results)} messages in {time.time() - start:.0f} seconds")

# 4. Intents and accuracy
description_to_intent = {d: i for i, d in descriptions.items()}
zero_shot_pred = [description_to_intent[r['labels'][0]] for r in results]
zero_shot_accuracy = accuracy_score(test['category'], zero_shot_pred)
print(f'Zero-shot accuracy: {zero_shot_accuracy:.3f}')

400 messages in 15 seconds
Zero-shot accuracy: 0.750


**Checkpoint:** the 400 messages take a few seconds to a minute, and the accuracy is about 0.76 with these descriptions (your own descriptions for the last three intents will move it a few points).

In [6]:
# 5. Which intents does the zero-shot model confuse? Show the crosstab of true and predicted intents.
pd.crosstab(
    test['category'].to_numpy(),
    np.array(zero_shot_pred),
    rownames=['true'], 
    colnames=['predicted']
)

predicted,card_arrival,card_delivery_estimate,card_not_working,change_pin,compromised_card,exchange_rate,lost_or_stolen_card,pin_blocked,top_up_failed,top_up_reverted
true,,,,,,,,,,
card_arrival,18,15,0,0,1,1,5,0,0,0
card_delivery_estimate,16,14,1,0,0,8,0,1,0,0
card_not_working,0,0,40,0,0,0,0,0,0,0
change_pin,0,0,0,40,0,0,0,0,0,0
compromised_card,1,0,2,0,26,1,10,0,0,0
exchange_rate,0,0,0,0,0,40,0,0,0,0
lost_or_stolen_card,3,0,3,0,2,1,31,0,0,0
pin_blocked,0,0,6,4,0,0,0,30,0,0
top_up_failed,0,0,2,0,0,0,0,0,38,0


**Question:** which intents does the zero-shot model handle well, and which does it confuse? Why is that hard to fix with better descriptions?

The zero-shot model handles intents with distinctive words well, such as `exchange_rate`, `compromised_card`, `top_up_reverted`. It confuses intents that share most of their vocabulary, for example `card_arrival` and `card_delivery_estimate`. Those intents differ in subtle ways and customers don’t word their messages like the descriptions. The model only compares each message with one short sentence and has never seen the bank’s labelled examples, so better descriptions mostly move the errors from one intent to its neighbour instead of removing them.

## Challenge 3: fine-tune DistilBERT

Now use the bank's 1,185 labelled messages. Follow the steps of section 6 of the lesson; the skeleton below has the structure, and you fill in the `TODO` parts.

1. Map every intent to a number (`label2id`) and back (`id2label`).
2. Write `to_dataset(df)`: build a `datasets.Dataset` with the columns `text` and `label`, and `map` the tokenizer on it with `truncation=True, max_length=64`.
3. Load `AutoModelForSequenceClassification` with 10 labels, `id2label` and `label2id`.
4. Train for 3 epochs with a batch size of 16 and a learning rate of `5e-5`, evaluating after every epoch.

In [7]:
# 1. Intents as numbers
labels = sorted(intents)
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}

# 2. Datasets
def to_dataset(df):
    data = datasets.Dataset.from_dict({
        'text': df['text'].tolist(),
        'label': [label2id[c] for c in df['category']] 
        })
    return data.map(lambda batch: tokenizer(batch['text'], truncation=True, max_length=128), batched=True)

train_data, test_data = to_dataset(train), to_dataset(test)

# 3. Model with a new classification layer
classifier = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels = len(labels),
    id2label = id2label,
    label2id = label2id
)

def compute_metrics(prediction):
    predicted = prediction.predictions.argmax(axis=-1)
    return {"accuracy": accuracy_score(prediction.label_ids, predicted)}

# 4. Training
args = TrainingArguments(
    output_dir="intent-classifier",
    num_train_epochs=3,              
    per_device_train_batch_size=16,
    learning_rate=5e-5,            
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="no",
    report_to="none",
    disable_tqdm=True,
    dataloader_pin_memory=False,
    seed=0,
)
trainer = Trainer(
    model = classifier,
    args = args,
    train_dataset = train_data,
    eval_dataset = test_data,
    processing_class = tokenizer,
    compute_metrics = compute_metrics
)

start = time.time()
trainer.train()
print(f"Training took {time.time() - start:.0f} seconds")

{'loss': '1.359', 'grad_norm': '3.847', 'learning_rate': '3.356e-05', 'epoch': '1'}
{'eval_loss': '0.5673', 'eval_accuracy': '0.915', 'eval_runtime': '2.467', 'eval_samples_per_second': '162.1', 'eval_steps_per_second': '20.26', 'epoch': '1'}
{'loss': '0.3769', 'grad_norm': '10.8', 'learning_rate': '1.689e-05', 'epoch': '2'}
{'eval_loss': '0.2737', 'eval_accuracy': '0.94', 'eval_runtime': '1.729', 'eval_samples_per_second': '231.3', 'eval_steps_per_second': '28.91', 'epoch': '2'}
{'loss': '0.1807', 'grad_norm': '10.3', 'learning_rate': '2.222e-07', 'epoch': '3'}
{'eval_loss': '0.2294', 'eval_accuracy': '0.94', 'eval_runtime': '1.711', 'eval_samples_per_second': '233.7', 'eval_steps_per_second': '29.22', 'epoch': '3'}
{'train_runtime': '101', 'train_samples_per_second': '35.21', 'train_steps_per_second': '2.229', 'train_loss': '0.6388', 'epoch': '3'}
Training took 101 seconds


In [9]:
# 5. Show the loss, the test loss and the test accuracy of every epoch (trainer.state.log_history)
history = pd.DataFrame(trainer.state.log_history)
history.groupby('epoch').first()[['loss', 'eval_loss', 'eval_accuracy']].round(3)

,loss,eval_loss,eval_accuracy
epoch,,,
1.0,1.359,0.567,0.915
2.0,0.377,0.274,0.940
3.0,0.181,0.229,0.940


**Checkpoint:** training takes one to four minutes on a CPU (a few seconds on a GPU). The test accuracy is about 0.92 after the first epoch and about 0.94 after the third, and both the training loss and the test loss fall at every epoch.

**Question:** the classification layer started with random weights. Why does the model still reach a good accuracy after a single pass over 1,185 messages?

It maps the 768-number [CLS] vector to 10 intents, about 7,700 weights against 66 million pretrained ones. The pretrained layers already turn each message into a vector that captures its meaning, so messages with different intents are already far apart, and the new layer only has to learn where to draw the boundaries. Fine-tuning with a small learning rate then adjusts the pretrained weights slightly toward the bank’s intents. One epoch is 75 updates, which is enough for such a small layer on top of good features.

## Challenge 4: compare with yesterday's semantic search

Yesterday's semantic search can also classify: give a new message the intent of the most similar past message. Use it as the baseline.

1. Encode the training and test messages with `SentenceTransformer("all-MiniLM-L6-v2")` and `normalize_embeddings=True`.
2. For each test message, take the intent of the training message with the highest dot product, and compute the accuracy.
3. Get the predictions of the fine-tuned model on the test set (`trainer.predict(test_data)`, then `argmax` and `id2label`).
4. Build a DataFrame with one row per method (zero-shot, semantic search, fine-tuned DistilBERT), with the accuracy and the number of labelled messages each one used.

In [10]:
# 1. Embeddings
minilm = SentenceTransformer('all-MiniLM-L6-v2')
train_vectors = minilm.encode(train['text'].tolist(), normalize_embeddings=True)
test_vectors = minilm.encode(test['text'].tolist(), normalize_embeddings=True)

# 2. Semantic search as a classifier
similarities = test_vectors @ train_vectors.T
best = similarities.argmax(axis=1)
search_pred = train['category'].to_numpy()[best]
search_accuracy = accuracy_score(test['category'], search_pred)

# 3. Fine-tuned predictions
fine_tuned_ids = trainer.predict(test_data).predictions.argmax(axis=-1)
fine_tuned_pred = [id2label[i] for i in fine_tuned_ids]
fine_tuned_accuracy = accuracy_score(test['category'], fine_tuned_pred)

# 4. Comparison table
comparison = pd.DataFrame({
    'labelled messages': [0, len(train), len(train)],
    'accuracy': [zero_shot_accuracy, search_accuracy, fine_tuned_accuracy]
}, index=['Zero-shot', 'Semantic Search', 'Fine-tuned DistilBERT'])
comparison.round(3)

,labelled messages,accuracy
Zero-shot,0,0.750
Semantic Search,1185,0.915
Fine-tuned DistilBERT,1185,0.940


**Checkpoint:** zero-shot about 0.76, semantic search about 0.92, fine-tuned DistilBERT about 0.94. Your fine-tuned accuracy can differ by a point or two from run to run and from computer to computer.

In [11]:
# 5. Print the classification report of the fine-tuned model, and the 6 most frequent (true, predicted) mistakes
print(classification_report(test["category"], fine_tuned_pred, digits=3))

errors = pd.DataFrame({"text": test["text"], "true": test["category"], "predicted": fine_tuned_pred})
errors = errors[errors["true"] != errors["predicted"]]
errors[["true", "predicted"]].value_counts().head(6)

                        precision    recall  f1-score   support

          card_arrival      0.860     0.925     0.892        40
card_delivery_estimate      0.923     0.900     0.911        40
      card_not_working      0.974     0.950     0.962        40
            change_pin      0.976     1.000     0.988        40
      compromised_card      0.950     0.950     0.950        40
         exchange_rate      1.000     1.000     1.000        40
   lost_or_stolen_card      0.949     0.925     0.937        40
           pin_blocked      0.974     0.950     0.962        40
         top_up_failed      0.848     0.975     0.907        40
       top_up_reverted      0.971     0.825     0.892        40

              accuracy                          0.940       400
             macro avg      0.943     0.940     0.940       400
          weighted avg      0.943     0.940     0.940       400



true                    predicted             
top_up_reverted         top_up_failed             7
card_delivery_estimate  card_arrival              4
card_arrival            card_delivery_estimate    3
lost_or_stolen_card     card_arrival              2
compromised_card        lost_or_stolen_card       2
card_not_working        compromised_card          1
Name: count, dtype: int64

In [12]:
errors.sort_values(["true", "predicted"])

,text,true,predicted
5,When will I get my card?,card_arrival,card_delivery_estimate
11,How long does a card delivery take?,card_arrival,card_delivery_estimate
32,How do I know when my card will arrive?,card_arrival,card_delivery_estimate
81,my card was not in the mail again can you advise?,card_delivery_estimate,card_arrival
84,I'm just wondering when my card will get here.,card_delivery_estimate,card_arrival
111,When can I expect to receive my new card?,card_delivery_estimate,card_arrival
112,I am waiting for my card to arrive.,card_delivery_estimate,card_arrival
127,Can you please identify the problem with my bank card?,card_not_working,compromised_card
128,How do I unblock my card using the app?,card_not_working,pin_blocked
284,Can I freeze my card right now?,compromised_card,lost_or_stolen_card


**Question:** where does the fine-tuned model still make mistakes? Are they model errors, or messages that a person would also find ambiguous?

The fine-tuned model’s remaining mistakes are concentrated in a few pairs of intents that share most of their vocabulary, `top_up_reverted`, `top_up_failed`. Reading the messages, most of these errors are ambiguous, eg. `When will I get my card?` a person could reasonably pick either intent. Only a few are model mistakes, eg. `Can you please identify the problem with my bank card?`.

## Challenge 5: route new messages and recommend a method

1. Call `trainer.model.eval()`, then wrap the fine-tuned model in a `text-classification` pipeline (`model=trainer.model, tokenizer=tokenizer`).
2. Route five new messages: four that you write yourself, one for each of four different intents, plus `"I don't know what happened, the app just says error"`. Print each message with its predicted intent and score.

In [13]:
# 1. Pipeline
trainer.model.eval()
intent_router = pipeline("text-classification", model=trainer.model, tokenizer=tokenizer)

# 2. Five new messages
new_messages = [
    "I ordered my card three weeks ago and it still hasn't come.",               
    "I typed the wrong PIN too many times and now it's blocked.",                  
    "What rate do you use when I pay in dollars with my euro account?",             
    "I tried to add money from my bank account but the top-up didn't go through.",   
    "I don't know what happened, the app just says error",
]

for message, result in zip(new_messages, intent_router(new_messages)):
    print(f'{result['label']:<24} {result['score']:.2f}  {message}')

card_arrival             0.93  I ordered my card three weeks ago and it still hasn't come.
pin_blocked              0.96  I typed the wrong PIN too many times and now it's blocked.
exchange_rate            0.92  What rate do you use when I pay in dollars with my euro account?
top_up_failed            0.94  I tried to add money from my bank account but the top-up didn't go through.
card_not_working         0.63  I don't know what happened, the app just says error


**Checkpoint:** the messages that clearly belong to one intent get the right one with a high score (0.9 or more for the card that has not arrived and for the blocked PIN). Messages that are vague, or that sit between intents, get a lower score: in the reference solution, the unknown payment in a shop goes to `card_not_working` with a score of only 0.26, and the app error message, which belongs to none of the ten intents, is still forced into one of them.

**Question:** the bank asks you which method to put in production. Write a short recommendation (4 to 6 sentences): which method, why, and what should happen to messages like the last one.

I would go with the fine-tuned DistilBERT, since it's the most accurate of the three methods (0.94) and it routes messages pretty quickly. Zero-shot is not accurate enough for routing without labels, but semantic search is a good fallback, especially since it doesn't require training. The model always picks one of the ten intents, even for a message like “the app just says error” that fits none of them, so its score should be used as a safety check. Messages with a score below a threshold should  be reviewed by a person and not routed automatically. Over time, adding new different intents and retrain the model regularly with the messages that agents corrected would lead to better and more accurate results.

## Extra: how many labels do you need?

Labelling messages costs the bank time. Find out how the fine-tuned model and the semantic search behave with fewer labels.

1. Write a function that keeps `n` random training messages per intent (`train.groupby("category").sample(n, random_state=0)`).
2. For `n` in 10 and 40, fine-tune a fresh DistilBERT on that subset (same settings as Challenge 3, but 6 epochs for `n=10` because there are so few batches) and compute the test accuracy. Run the semantic search with the same subset as the past messages.
3. Show the results in a table, and add the numbers of Challenge 4 for the full training set.

In [ ]:
# Your code here

**Conclusions:** *Your answer here.*